# Agente de análise de estoque (LangGraph + Ollama)

Fluxo: **pergunta → `interpretar` (LLM) → decisão → `consultar_api` (ferramenta) → `responder`**, ou direto para `responder_direto` quando a API não é necessária.

Cadastro: **`interpretar` → `preparar_cadastro` (LLM extrai os dados) → `completar_cadastro` (pergunta o que faltar) → `confirmar_cadastro` (confirmação humana) → `responder_cadastro`**.

Pré-requisitos:
- API de estoque no ar em `http://127.0.0.1:8000` (repositório `api-estoque`).
- Ollama rodando com o modelo `qwen2.5:7b` (local e gratuito).
- Kernel do notebook: o Python do `.venv` deste projeto.

## 1. Configuração

In [1]:
import os
import re
import uuid
from typing import Literal, Optional, TypedDict

import httpx
from dotenv import load_dotenv
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel, ConfigDict, Field, StrictInt, ValidationError

load_dotenv()  # lê o .env, se existir (ele não é versionado)
API_URL = os.getenv("API_ESTOQUE_URL", "http://127.0.0.1:8000")
MODELO = os.getenv("OLLAMA_MODEL", "qwen2.5:7b")

# temperature=0 deixa as respostas estáveis; timeout evita ficar preso se o Ollama travar
llm = ChatOllama(model=MODELO, temperature=0, client_kwargs={"timeout": 120})
print(f"Configuração realizada com sucesso (API: {API_URL} | modelo: {MODELO})")

Configuração realizada com sucesso (API: http://127.0.0.1:8000 | modelo: qwen2.5:7b)


## 2. Ferramentas: `consultar_produto` e `cadastrar_produto`

**`consultar_produto`**
- **Finalidade:** consultar um produto na API de estoque (`GET /produtos/{codigo}`).
- **Parâmetro:** `codigo` no formato `PRD-XXX`.
- **Retorno:** `{"ok": True, "produto": {...}}` ou `{"ok": False, "erro": ..., "mensagem": ...}`.
- **Falhas tratadas:** código inválido (validado antes da chamada), produto inexistente (404), serviço indisponível (503), API desligada/timeout e resposta com dados inválidos.

**`cadastrar_produto`** (altera dados: só é chamada após confirmação humana)
- **Finalidade:** cadastrar um novo produto (`POST /produtos`).
- **Parâmetro:** `produto` com `codigo`, `nome`, `categoria`, `quantidade_atual`, `estoque_minimo` e `unidade` (todos obrigatórios).
- **Retorno:** o mesmo formato da consulta.
- **Falhas tratadas:** dados inválidos (validados antes da chamada, com as mesmas regras da API), código duplicado (409), dados recusados pela API (422) e API desligada/timeout.

In [2]:
CODIGO_VALIDO = re.compile(r"^PRD-[0-9]{3}$")
CAMPOS_OBRIGATORIOS = {"codigo", "nome", "quantidade_atual", "estoque_minimo"}
API_INACESSIVEL = {"ok": False, "erro": "api_inacessivel",
                   "mensagem": "Não foi possível conectar à API de estoque. Verifique se ela está no ar."}


def _corpo_json(r: httpx.Response) -> dict:
    try:
        corpo = r.json()
    except ValueError:
        return {}
    return corpo if isinstance(corpo, dict) else {}


@tool
def consultar_produto(codigo: str) -> dict:
    """Consulta um produto na API de estoque pelo código (formato PRD-XXX, ex.: PRD-001).

    Retorna {"ok": True, "produto": {...}} ou {"ok": False, "erro": <identificador>, "mensagem": <texto>}.
    """
    codigo = codigo.strip().upper()
    if not CODIGO_VALIDO.match(codigo):
        return {"ok": False, "erro": "codigo_invalido",
                "mensagem": f"O código '{codigo}' não segue o formato PRD-XXX (ex.: PRD-001)."}

    try:
        r = httpx.get(f"{API_URL}/produtos/{codigo}", timeout=5)
    except httpx.HTTPError:
        return API_INACESSIVEL

    corpo = _corpo_json(r)
    if r.status_code != 200:
        return {"ok": False, "erro": corpo.get("erro", f"http_{r.status_code}"),
                "mensagem": corpo.get("mensagem", "Erro inesperado da API.")}

    # Verifica o retorno antes de usar: campos presentes e quantidades inteiras >= 0
    quantidades_ok = all(isinstance(corpo.get(c), int) and corpo[c] >= 0
                         for c in ("quantidade_atual", "estoque_minimo"))
    if not (CAMPOS_OBRIGATORIOS <= corpo.keys() and quantidades_ok):
        return {"ok": False, "erro": "resposta_invalida",
                "mensagem": "A API retornou dados incompletos ou inválidos."}
    return {"ok": True, "produto": corpo}


class NovoProduto(BaseModel):
    """Corpo do POST /produtos, com as mesmas regras da API (todos os campos obrigatórios)."""
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)

    codigo: str = Field(pattern=r"^PRD-[0-9]{3}$")
    nome: str = Field(min_length=1)
    categoria: str = Field(min_length=1)
    quantidade_atual: StrictInt = Field(ge=0)
    estoque_minimo: StrictInt = Field(ge=0)
    unidade: str = Field(min_length=1)


def validar_novo_produto(dados: dict) -> dict:
    """Retorna {"ok": True, "produto": {...}} ou {"ok": False, "erro": "dados_invalidos", "mensagem": ...}."""
    try:
        produto = NovoProduto.model_validate(dados).model_dump()
    except ValidationError as e:
        problemas = "; ".join(f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in e.errors())
        return {"ok": False, "erro": "dados_invalidos", "mensagem": f"Dados inválidos para cadastro ({problemas})."}
    if produto["codigo"] == "PRD-503":
        return {"ok": False, "erro": "dados_invalidos",
                "mensagem": "O código PRD-503 é reservado pela API e não pode ser cadastrado."}
    return {"ok": True, "produto": produto}


@tool
def cadastrar_produto(produto: dict) -> dict:
    """Cadastra um novo produto na API de estoque (POST /produtos). Use somente após confirmação humana.

    produto: codigo (PRD-XXX), nome, categoria, quantidade_atual (inteiro >= 0), estoque_minimo (inteiro >= 0), unidade.
    Retorna {"ok": True, "produto": {...}} ou {"ok": False, "erro": <identificador>, "mensagem": <texto>}.
    """
    validacao = validar_novo_produto(produto)
    if not validacao["ok"]:
        return validacao

    try:
        r = httpx.post(f"{API_URL}/produtos", json=validacao["produto"], timeout=5)
    except httpx.HTTPError:
        return API_INACESSIVEL

    corpo = _corpo_json(r)
    if r.status_code != 201:  # 409 produto_duplicado, 422 dados_invalidos
        return {"ok": False, "erro": corpo.get("erro", f"http_{r.status_code}"),
                "mensagem": corpo.get("mensagem", "Erro inesperado da API.")}
    return {"ok": True, "produto": corpo}

print("Ferramentas consultar_produto e cadastrar_produto criadas com sucesso")

Ferramentas consultar_produto e cadastrar_produto criadas com sucesso


## 3. Regra de situação do estoque

Calculada em Python (não pelo LLM), para o resultado ser sempre correto e verificável:

| Condição | Situação |
|---|---|
| `quantidade_atual == 0` | esgotado |
| `0 < quantidade_atual <= estoque_minimo` | baixo (inclui estar exatamente no mínimo) |
| `quantidade_atual > estoque_minimo` | disponível |

In [3]:
def classificar_situacao(quantidade: int, minimo: int) -> str:
    if quantidade == 0:
        return "esgotado"
    if quantidade <= minimo:
        return "baixo"
    return "disponível"

print("Regra classificar_situacao definida com sucesso")

Regra classificar_situacao definida com sucesso


## 4. Instruções do agente (papel, objetivo, contexto e restrições)

In [4]:
PROMPT_SISTEMA = """Papel: você é um assistente de análise de estoque de uma loja.
Objetivo: informar a disponibilidade e a situação do estoque de produtos.
Contexto: os dados vêm exclusivamente da API de estoque. Cada produto tem código PRD-XXX,
quantidade_atual e estoque_minimo. Situações: esgotado (quantidade 0), baixo (quantidade maior
que 0 e menor ou igual ao mínimo) e disponível (acima do mínimo).
O estoque mínimo é o ponto de reposição: quando a quantidade atual chega ao mínimo ou fica abaixo dele,
o produto precisa ser reposto para não esgotar.
Restrições:
- Use somente os dados fornecidos; nunca invente produtos, códigos ou quantidades.
- Você consulta produtos e cadastra novos somente após confirmação humana; não altera nem exclui produtos.
- Ao falar de um produto, informe sempre a quantidade atual e o estoque mínimo.
- Responda em português, de forma breve e objetiva."""

PROMPT_CLASSIFICACAO = """Classifique a mensagem do usuário em uma intenção:
- consultar_produto: pede estoque, quantidade, disponibilidade ou situação de um produto e informa um código
  (copie o código exatamente como o usuário escreveu no campo codigo, mesmo que pareça errado).
- conceitual: pergunta geral sobre conceitos de estoque (ex.: o que é estoque mínimo), sem pedir dados de um produto.
- ambigua: quer dados de produto, mas não informou nenhum código (nome de produto, como mouse ou teclado, não é código).
- cadastrar_produto: pede para cadastrar, incluir ou criar um novo produto.
- fora_do_escopo: qualquer outro assunto, inclusive pedidos para alterar ou excluir produtos.

Exemplos:
- "Qual a situação do PRD-001?" -> consultar_produto, codigo PRD-001
- "Tem o item ABC-1 em estoque?" -> consultar_produto, codigo ABC-1
- "O que significa estoque baixo?" -> conceitual
- "Como está o estoque do mouse?" -> ambigua
- "Cadastre o produto PRD-020, Teclado, categoria Periféricos, 15 unidades, mínimo 5" -> cadastrar_produto, codigo PRD-020
- "Exclua o produto PRD-001" -> fora_do_escopo
- "Altere a quantidade do PRD-002 para 30" -> fora_do_escopo
- "Qual a capital da França?" -> fora_do_escopo"""

PROMPT_EXTRACAO = """Extraia os dados do produto que o usuário quer cadastrar.
Campos: codigo, nome, categoria, quantidade_atual, estoque_minimo, unidade.
- Copie os valores exatamente como o usuário escreveu, inclusive números negativos.
- Nunca invente: deixe nulo o campo que o usuário não informou."""

print("Prompts do agente definidos")

Prompts do agente definidos


## 5. Estado compartilhado e nós do grafo

- `interpretar`: o LLM classifica a intenção e extrai o código (saída estruturada).
- `decidir_rota`: **transição condicional** — usa a ferramenta só quando há produto a consultar.
- `consultar_api`: chama a ferramenta e calcula a situação.
- `responder`: o LLM redige a resposta com base **apenas** nos dados da API; em falha, mensagem fixa (sem LLM, para não inventar dados).
- `responder_direto`: perguntas conceituais (LLM, sem API) ou mensagens fixas para ambígua/fora do escopo.
- `preparar_cadastro`: o LLM extrai os dados do novo produto (saída estruturada). Só são aceitos valores que aparecem no texto do usuário (valor inventado ou alterado pelo modelo é descartado). Dados inválidos encerram o fluxo **antes** da confirmação.
- `completar_cadastro`: se faltar algum campo, pergunta ao usuário (`interrupt()`); uma única rodada, o que continuar faltando vira erro de validação.
- `confirmar_cadastro`: **confirmação humana** com `interrupt()` do LangGraph; só chama `cadastrar_produto` (POST) se a resposta for "s".
- `responder_cadastro`: mensagem fixa com o resultado do cadastro (sem LLM, para não inventar dados).

In [5]:
class Decisao(BaseModel):
    intencao: Literal["consultar_produto", "cadastrar_produto", "conceitual", "ambigua", "fora_do_escopo"]
    codigo: Optional[str] = Field(None, description="Código do produto citado, exatamente como escrito pelo usuário")


class DadosCadastro(BaseModel):
    """Saída do LLM no cadastro: campo não informado fica nulo (a validação acusa a falta)."""
    codigo: Optional[str] = None
    nome: Optional[str] = None
    categoria: Optional[str] = None
    quantidade_atual: Optional[int] = None
    estoque_minimo: Optional[int] = None
    unidade: Optional[str] = None


class Estado(TypedDict, total=False):
    pergunta: str
    intencao: str
    codigo: Optional[str]
    dados_cadastro: dict     # produto extraído da pergunta, aguardando confirmação
    resultado: dict          # retorno da ferramenta
    situacao: Optional[str]
    resposta: str


MENSAGENS_FIXAS = {
    "ambigua": "Não identifiquei qual produto você quer consultar. Informe o código no formato PRD-XXX (ex.: PRD-001).",
    "fora_do_escopo": "Só posso consultar o estoque e cadastrar novos produtos (com sua confirmação). "
                      "Não altero nem excluo produtos.",
    "falha_llm": "Não consegui processar a solicitação agora (modelo local indisponível). "
                 "Verifique se o Ollama está rodando e tente novamente.",
}

classificador = llm.with_structured_output(Decisao)
extrator = llm.with_structured_output(DadosCadastro)


CODIGO_NO_TEXTO = re.compile(r"\bPRD-[0-9]{3}\b", re.IGNORECASE)


def interpretar(estado: Estado) -> Estado:
    try:
        d = classificador.invoke([("system", PROMPT_CLASSIFICACAO), ("human", estado["pergunta"])])
        # O LLM decide a intenção; o código PRD-XXX é extraído por regex (modelos pequenos às vezes o omitem)
        achado = CODIGO_NO_TEXTO.search(estado["pergunta"])
        return {"intencao": d.intencao, "codigo": achado.group(0) if achado else d.codigo}
    except Exception as e:  # Ollama fora do ar, timeout ou saída fora do formato
        print(f"[interpretar] falha no LLM: {e}")
        return {"intencao": "falha_llm"}


def decidir_rota(estado: Estado) -> str:
    if estado["intencao"] == "consultar_produto" and estado.get("codigo"):
        return "consultar_api"
    if estado["intencao"] == "cadastrar_produto":
        return "preparar_cadastro"
    return "responder_direto"


def consultar_api(estado: Estado) -> Estado:
    resultado = consultar_produto.invoke({"codigo": estado["codigo"]})
    situacao = None
    if resultado["ok"]:
        p = resultado["produto"]
        situacao = classificar_situacao(p["quantidade_atual"], p["estoque_minimo"])
    return {"resultado": resultado, "situacao": situacao}


def responder(estado: Estado) -> Estado:
    r = estado["resultado"]
    if not r["ok"]:
        return {"resposta": f"Não consegui obter os dados do produto {estado['codigo']}: {r['mensagem']}"}

    p = r["produto"]
    resumo = (f"{p['nome']} ({p['codigo']}): {p['quantidade_atual']} {p.get('unidade', '')} em estoque, "
              f"estoque mínimo {p['estoque_minimo']}, situação: {estado['situacao']}.")
    try:
        msg = llm.invoke([
            ("system", PROMPT_SISTEMA),
            ("human", f"Pergunta: {estado['pergunta']}\nDados da API: {resumo}\n"
                      "Responda à pergunta usando apenas esses dados e cite a quantidade atual e o estoque mínimo."),
        ])
        # Garante a quantidade na resposta mesmo se o modelo a omitir (requisito do cenário de teste 1)
        if str(p["quantidade_atual"]) not in msg.content:
            return {"resposta": f"{msg.content}\n{resumo}"}
        return {"resposta": msg.content}
    except Exception as e:
        print(f"[responder] falha no LLM, usando resumo fixo: {e}")
        return {"resposta": resumo}


def responder_direto(estado: Estado) -> Estado:
    if estado["intencao"] == "conceitual":
        try:
            msg = llm.invoke([
                ("system", PROMPT_SISTEMA + "\nResponda de forma conceitual, sem citar produtos nem quantidades."),
                ("human", estado["pergunta"]),
            ])
            return {"resposta": msg.content}
        except Exception as e:
            print(f"[responder_direto] falha no LLM: {e}")
            return {"resposta": MENSAGENS_FIXAS["falha_llm"]}
    # consultar_produto sem código cai aqui e é tratado como ambígua
    return {"resposta": MENSAGENS_FIXAS.get(estado["intencao"], MENSAGENS_FIXAS["ambigua"])}


CAMPOS_CADASTRO = list(NovoProduto.model_fields)
FALHA_LLM_CADASTRO = {"resultado": {"ok": False, "erro": "falha_llm", "mensagem": MENSAGENS_FIXAS["falha_llm"]}}


def extrair_dados(texto: str) -> dict:
    """O LLM extrai os campos; só ficam os valores que aparecem no texto (evita dado inventado ou alterado)."""
    d = extrator.invoke([("system", PROMPT_EXTRACAO), ("human", texto)])
    numeros = set(re.findall(r"-?\d+", texto))
    dados = {}
    for campo, valor in d.model_dump().items():
        if valor is None:
            continue
        presente = str(valor) in numeros if isinstance(valor, int) else str(valor).strip().lower() in texto.lower()
        if presente:
            dados[campo] = valor
    return dados


def faltando(dados: dict) -> list:
    return [c for c in CAMPOS_CADASTRO if c not in dados]


def validar_cadastro(dados: dict) -> Estado:
    validacao = validar_novo_produto(dados)
    if not validacao["ok"]:
        return {"dados_cadastro": dados, "resultado": validacao}
    return {"dados_cadastro": validacao["produto"]}


def preparar_cadastro(estado: Estado) -> Estado:
    try:
        dados = extrair_dados(estado["pergunta"])
    except Exception as e:
        print(f"[preparar_cadastro] falha no LLM: {e}")
        return FALHA_LLM_CADASTRO
    if estado.get("codigo"):  # código copiado do texto pela regex, como na consulta
        dados["codigo"] = estado["codigo"].upper()
    if faltando(dados):
        return {"dados_cadastro": dados}  # completar_cadastro pergunta o que falta
    return validar_cadastro(dados)


def completar_cadastro(estado: Estado) -> Estado:
    dados = dict(estado["dados_cadastro"])
    faltam = faltando(dados)
    resposta = interrupt(f"Para cadastrar, informe: {', '.join(faltam)} (ex.: estoque mínimo 5, unidade un).")
    try:
        novos = extrair_dados(str(resposta))
    except Exception as e:
        print(f"[completar_cadastro] falha no LLM: {e}")
        return FALHA_LLM_CADASTRO
    dados.update({c: v for c, v in novos.items() if c in faltam})
    return validar_cadastro(dados)  # rodada única: o que ainda faltar vira erro de validação


def decidir_cadastro(estado: Estado) -> str:
    if "resultado" in estado:  # erro (dados inválidos ou falha do LLM): encerra sem pedir confirmação
        return "responder_cadastro"
    if faltando(estado["dados_cadastro"]):
        return "completar_cadastro"
    return "confirmar_cadastro"


def confirmar_cadastro(estado: Estado) -> Estado:
    p = estado["dados_cadastro"]
    resposta = interrupt(f"Confirma o cadastro de {p['codigo']} - {p['nome']} ({p['categoria']}), "
                         f"{p['quantidade_atual']} {p['unidade']}, estoque mínimo {p['estoque_minimo']}? (s/n)")
    # Ao retomar, o nó roda de novo e interrupt() devolve a resposta: o POST só acontece depois dela
    if str(resposta).strip().lower() not in ("s", "sim"):
        return {"resultado": {"ok": False, "erro": "cancelado", "mensagem": "cadastro cancelado pelo usuário."}}
    return {"resultado": cadastrar_produto.invoke({"produto": p})}


def responder_cadastro(estado: Estado) -> Estado:
    r = estado["resultado"]
    if not r["ok"]:
        return {"resposta": f"Não foi possível cadastrar o produto: {r['mensagem']}"}
    p = r["produto"]
    situacao = classificar_situacao(p["quantidade_atual"], p["estoque_minimo"])
    return {"situacao": situacao,
            "resposta": f"Produto {p['codigo']} ({p['nome']}) cadastrado com sucesso: {p['quantidade_atual']} "
                        f"{p['unidade']} em estoque, estoque mínimo {p['estoque_minimo']}, situação: {situacao}."}

print("Estado e nós do grafo definidos")

Estado e nós do grafo definidos


## 6. Montagem do grafo

In [6]:
grafo = StateGraph(Estado)
grafo.add_node("interpretar", interpretar)
grafo.add_node("consultar_api", consultar_api)
grafo.add_node("responder", responder)
grafo.add_node("responder_direto", responder_direto)
grafo.add_node("preparar_cadastro", preparar_cadastro)
grafo.add_node("completar_cadastro", completar_cadastro)
grafo.add_node("confirmar_cadastro", confirmar_cadastro)
grafo.add_node("responder_cadastro", responder_cadastro)

grafo.add_edge(START, "interpretar")
grafo.add_conditional_edges("interpretar", decidir_rota, ["consultar_api", "preparar_cadastro", "responder_direto"])
ROTAS_CADASTRO = ["completar_cadastro", "confirmar_cadastro", "responder_cadastro"]
grafo.add_conditional_edges("preparar_cadastro", decidir_cadastro, ROTAS_CADASTRO)
grafo.add_conditional_edges("completar_cadastro", decidir_cadastro, ROTAS_CADASTRO[1:])  # rodada única, sem laço
grafo.add_edge("confirmar_cadastro", "responder_cadastro")
grafo.add_edge("consultar_api", "responder")
grafo.add_edge("responder", END)          # condição de encerramento
grafo.add_edge("responder_direto", END)   # condição de encerramento
grafo.add_edge("responder_cadastro", END) # condição de encerramento

agente = grafo.compile(checkpointer=InMemorySaver())  # o checkpointer permite pausar na confirmação humana

print(agente.get_graph().draw_mermaid())  # diagrama em Mermaid
print("Grafo compilado com sucesso")

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	interpretar(interpretar)
	consultar_api(consultar_api)
	responder(responder)
	responder_direto(responder_direto)
	preparar_cadastro(preparar_cadastro)
	completar_cadastro(completar_cadastro)
	confirmar_cadastro(confirmar_cadastro)
	responder_cadastro(responder_cadastro)
	__end__([<p>__end__</p>]):::last
	__start__ --> interpretar;
	completar_cadastro -.-> confirmar_cadastro;
	completar_cadastro -.-> responder_cadastro;
	confirmar_cadastro --> responder_cadastro;
	consultar_api --> responder;
	interpretar -.-> consultar_api;
	interpretar -.-> preparar_cadastro;
	interpretar -.-> responder_direto;
	preparar_cadastro -.-> completar_cadastro;
	preparar_cadastro -.-> confirmar_cadastro;
	preparar_cadastro -.-> responder_cadastro;
	responder --> __end__;
	responder_cadastro --> __end__;
	responder_direto --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	c

## 7. Execução

`perguntar()` valida a entrada, executa o grafo com `recursion_limit` (parada garantida mesmo se o fluxo mudar) e mostra o estado final, que serve como evidência.

No cadastro, o grafo pausa para pedir ao humano os campos que faltarem e a confirmação (s/n). Sem o parâmetro `respostas`, cada resposta é pedida com `input()`; nas demonstrações abaixo elas são passadas em `respostas` (na ordem das perguntas) para o notebook rodar sem travar.

In [7]:
LIMITE_PERGUNTA = 500


def perguntar(pergunta: str, respostas: Optional[list] = None) -> Estado:
    pergunta = (pergunta or "").strip()
    if not pergunta or len(pergunta) > LIMITE_PERGUNTA:
        print(f"Entrada inválida: a pergunta deve ter entre 1 e {LIMITE_PERGUNTA} caracteres.")
        return {}

    config = {"recursion_limit": 10, "configurable": {"thread_id": str(uuid.uuid4())}}
    estado = agente.invoke({"pergunta": pergunta}, config=config)
    respostas = list(respostas or [])
    while estado.get("__interrupt__"):  # o agente pede algo ao humano (campos faltantes ou confirmação)
        pedido = estado["__interrupt__"][0].value
        resposta = respostas.pop(0) if respostas else input(pedido + " ")
        print(f"Agente   : {pedido}\nHumano   : {resposta}")
        estado = agente.invoke(Command(resume=resposta), config=config)
    print(f"Pergunta : {pergunta}")
    print(f"Intenção : {estado.get('intencao')} | código: {estado.get('codigo')} | situação: {estado.get('situacao')}")
    print(f"API      : {estado.get('resultado', 'não consultada')}")
    print(f"Resposta : {estado['resposta']}\n")
    return estado

print("Função perguntar pronta para uso")

Função perguntar pronta para uso


### Produto existente (baixo, esgotado e disponível)

In [8]:
perguntar("Qual a situação do estoque do produto PRD-001?")
perguntar("Ainda tem PRD-004 em estoque?")
perguntar("Como está o PRD-002?");

Pergunta : Qual a situação do estoque do produto PRD-001?
Intenção : consultar_produto | código: PRD-001 | situação: baixo
API      : {'ok': True, 'produto': {'codigo': 'PRD-001', 'nome': 'Mouse sem fio', 'categoria': 'Periféricos', 'quantidade_atual': 5, 'estoque_minimo': 10, 'unidade': 'un'}}
Resposta : O produto PRD-001 está em situação de baixo estoque. A quantidade atual é de 5 unidades, enquanto o estoque mínimo é de 10 unidades.

Pergunta : Ainda tem PRD-004 em estoque?
Intenção : consultar_produto | código: PRD-004 | situação: esgotado
API      : {'ok': True, 'produto': {'codigo': 'PRD-004', 'nome': 'Cabo HDMI 2 m', 'categoria': 'Cabos e Adaptadores', 'quantidade_atual': 0, 'estoque_minimo': 20, 'unidade': 'un'}}
Resposta : A quantidade atual de PRD-004 é 0 unidades, com estoque mínimo de 20 unidades. Situação: esgotado.

Pergunta : Como está o PRD-002?
Intenção : consultar_produto | código: PRD-002 | situação: disponível
API      : {'ok': True, 'produto': {'codigo': 'PRD-002',

### Produto inexistente e falhas da API

In [9]:
perguntar("Qual o estoque do PRD-999?")      # 404
perguntar("Consulte o produto PRD-503")       # 503 simulado
perguntar("Quantas unidades tem o ABC-1?");   # código inválido, barrado antes da chamada

Pergunta : Qual o estoque do PRD-999?
Intenção : consultar_produto | código: PRD-999 | situação: None
API      : {'ok': False, 'erro': 'produto_nao_encontrado', 'mensagem': 'Produto não encontrado.'}
Resposta : Não consegui obter os dados do produto PRD-999: Produto não encontrado.

Pergunta : Consulte o produto PRD-503
Intenção : consultar_produto | código: PRD-503 | situação: None
API      : {'ok': False, 'erro': 'servico_indisponivel', 'mensagem': 'Serviço temporariamente indisponível. Tente novamente mais tarde.'}
Resposta : Não consegui obter os dados do produto PRD-503: Serviço temporariamente indisponível. Tente novamente mais tarde.

Pergunta : Quantas unidades tem o ABC-1?
Intenção : consultar_produto | código: ABC-1 | situação: None
API      : {'ok': False, 'erro': 'codigo_invalido', 'mensagem': "O código 'ABC-1' não segue o formato PRD-XXX (ex.: PRD-001)."}
Resposta : Não consegui obter os dados do produto ABC-1: O código 'ABC-1' não segue o formato PRD-XXX (ex.: PRD-001).



### Sem consulta à API (conceitual, ambígua e fora do escopo)

In [10]:
perguntar("O que é estoque mínimo?")
perguntar("Como está o estoque do mouse?")
perguntar("Exclua o produto PRD-001");

Pergunta : O que é estoque mínimo?
Intenção : conceitual | código: None | situação: None
API      : não consultada
Resposta : O estoque mínimo é o nível de estoque abaixo do qual o produto precisa ser reabastecido para evitar esgotamento. Quando a quantidade atual atinge ou cai abaixo desse nível, indica que é necessário realizar uma reposição para manter a disponibilidade do produto na loja.

Pergunta : Como está o estoque do mouse?
Intenção : ambigua | código: None | situação: None
API      : não consultada
Resposta : Não identifiquei qual produto você quer consultar. Informe o código no formato PRD-XXX (ex.: PRD-001).

Pergunta : Exclua o produto PRD-001
Intenção : fora_do_escopo | código: PRD-001 | situação: None
API      : não consultada
Resposta : Só posso consultar o estoque e cadastrar novos produtos (com sua confirmação). Não altero nem excluo produtos.



### Cadastro de produto (POST /produtos, com confirmação humana e pedido dos campos faltantes)

In [11]:
perguntar("Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un",
          respostas=["s"])
perguntar("Qual a situação do estoque do PRD-020?")
perguntar("Quero cadastrar o PRD-030, uma Webcam HD da categoria Periféricos, com 8 unidades",
          respostas=["nome Webcam HD, estoque mínimo 3, unidade un", "s"]);  # frase incompleta: o agente pergunta o que falta

Agente   : Confirma o cadastro de PRD-020 - Teclado mecânico (Periféricos), 15 un, estoque mínimo 5? (s/n)
Humano   : s
Pergunta : Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un
Intenção : cadastrar_produto | código: PRD-020 | situação: disponível
API      : {'ok': True, 'produto': {'codigo': 'PRD-020', 'nome': 'Teclado mecânico', 'categoria': 'Periféricos', 'quantidade_atual': 15, 'estoque_minimo': 5, 'unidade': 'un'}}
Resposta : Produto PRD-020 (Teclado mecânico) cadastrado com sucesso: 15 un em estoque, estoque mínimo 5, situação: disponível.

Pergunta : Qual a situação do estoque do PRD-020?
Intenção : consultar_produto | código: PRD-020 | situação: disponível
API      : {'ok': True, 'produto': {'codigo': 'PRD-020', 'nome': 'Teclado mecânico', 'categoria': 'Periféricos', 'quantidade_atual': 15, 'estoque_minimo': 5, 'unidade': 'un'}}
Resposta : O produto PRD-020 (Teclado mecânico) está disponível, com 15 un

### Falhas no cadastro (código duplicado e dados inválidos)

In [ ]:
perguntar("Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un",
          respostas=["s"])  # Deve retornar 409: o PRD-020 já existe
perguntar("Cadastre o produto PRD-021: nome Mouse gamer, categoria Periféricos, quantidade atual -3, estoque mínimo 2, unidade un");  # barrado antes da confirmação

Agente   : Confirma o cadastro de PRD-020 - Teclado mecânico (Periféricos), 15 un, estoque mínimo 5? (s/n)
Humano   : s
Pergunta : Cadastre o produto PRD-020: nome Teclado mecânico, categoria Periféricos, quantidade atual 15, estoque mínimo 5, unidade un
Intenção : cadastrar_produto | código: PRD-020 | situação: None
API      : {'ok': False, 'erro': 'produto_duplicado', 'mensagem': 'Já existe um produto cadastrado com o código PRD-020.'}
Resposta : Não foi possível cadastrar o produto: Já existe um produto cadastrado com o código PRD-020.

Pergunta : Cadastre o produto PRD-021: nome Mouse gamer, categoria Periféricos, quantidade atual -3, estoque mínimo 2, unidade un
Intenção : cadastrar_produto | código: PRD-021 | situação: None
API      : {'ok': False, 'erro': 'dados_invalidos', 'mensagem': 'Dados inválidos para cadastro (quantidade_atual: Input should be greater than or equal to 0).'}
Resposta : Não foi possível cadastrar o produto: Dados inválidos para cadastro (quantidade_atual: I